# Shape Blend Splines — Interactive Demo

**Paper:** Q. Li, *Shape Blend Splines*, Computer-Aided Design, 2011. DOI: [10.1016/j.cad.2011.01.006](https://doi.org/10.1016/j.cad.2011.01.006)  
**Repository:** [QL-UoHull/Shape-Blend-Splines](https://github.com/QL-UoHull/Shape-Blend-Splines)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QL-UoHull/Shape-Blend-Splines/blob/main/notebooks/interactive_shape_blend_demo.ipynb)

---

This notebook demonstrates the **Shape Blend Spline (SBS)** framework:

- Blend any number of simple **parametric shapes** (circle, ellipse, rectangle, star, …) into a smooth complex curve.
- Compare **global weighted blending** helpers with **locality-aware SBS** examples.
- Control the **locality parameter** α in the SBS sections to tune how strongly each shape is preserved near its centre parameter.
- Adjust **per-shape weights** interactively with sliders.

The core idea:

$$\mathbf{C}(t) = \sum_{j=0}^{k-1} W_j(t)\, \mathbf{S}_j(t)$$

where $W_j(t)$ are *shape-preserving partition-of-unity* weights and $\mathbf{S}_j$ are the constituent shapes.


In [ ]:
# ── Install dependencies (only needed on Google Colab) ──────────────────────
import sys, subprocess, importlib

def colab_install():
    try:
        import google.colab  # only available on Colab
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "numpy", "matplotlib", "ipywidgets", "--quiet"])
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "git+https://github.com/QL-UoHull/Shape-Blend-Splines.git",
                               "--quiet"])
        print("✔  Dependencies installed.")
    except ImportError:
        pass  # not on Colab — assume local install

colab_install()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

from shape_blend_splines import ShapeBlendSpline, ShapeBlender, blend_two_shapes, blend_shape_series, shape_morph
from shape_blend_splines.shapes import (
    circle_arc, ellipse_arc, superellipse_arc,
    rectangle_arc, star_arc, line_segment, from_control_points
)
from shape_blend_splines.basis import blend_weights

# Use interactive matplotlib backend if available
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

print("✔  Imports OK")

## 1  Four-control-point shape design

A **Shape Blend Spline** treats each constituent shape as a *control element* — the direct analogue of a control point in a traditional Bézier or B-spline curve.  
Here we use **four control shapes** — an ellipse, a squircle (superellipse, n = 4), a 5-pointed star, and a square — each centred at the origin, and blend them into a single smooth closed curve.

The **locality parameter α** controls how strongly each region of the blended curve resembles its nearest control shape:

| α | Character |
|---|----------|
| Low (≈ 0.5) | Global blend — all four shapes are averaged into a featureless oval |
| Moderate (1–3) | Visible transitions — each region begins to reflect its control shape |
| High (≥ 5) | Strong preservation — each zone closely tracks its local control shape |

Increasing α is analogous to raising a B-spline degree: it sharpens the *locality* of the basis functions so each shape contributes only near its own centre parameter.


In [ ]:
from functools import partial
from shape_blend_splines import ShapeBlendSpline
from shape_blend_splines.shapes import ellipse_arc, superellipse_arc, star_arc, rectangle_arc

# ── Four control shapes (all centred at origin, comparable scale) ─────────
ctrl_shapes = [
    partial(ellipse_arc,      center=(0,0), a=1.3, b=0.7),
    partial(superellipse_arc, center=(0,0), a=1.0, b=1.0, exponent=4.0),
    partial(star_arc,         center=(0,0), outer_radius=1.1, inner_radius=0.45, n_points=5),
    partial(rectangle_arc,    center=(0,0), width=1.6, height=1.6),
]
ctrl_labels = ['① Ellipse', '② Squircle (n=4)', '③ 5-Star', '④ Square']
ctrl_colors = ['#4477AA', '#EE6677', '#228833', '#CCBB44']

alpha_values = [0.5, 1.5, 3.5, 7.0]
t = np.linspace(0, 1, 600)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, alpha in zip(axes, alpha_values):
    sbs = ShapeBlendSpline(ctrl_shapes, locality=alpha)
    pts = sbs.evaluate(t)

    # Component shapes (dashed)
    for s, lbl, col in zip(ctrl_shapes, ctrl_labels, ctrl_colors):
        sp = s(t)
        ax.plot(sp[:,0], sp[:,1], '--', color=col, lw=1.2, alpha=0.5, label=lbl)

    # SBS blend curve (solid black)
    ax.plot(pts[:,0], pts[:,1], 'k-', lw=2.8, zorder=5)
    ax.set_aspect('equal')
    ax.set_title(f'α = {alpha}', fontsize=12)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_xlim(-1.9, 1.9); ax.set_ylim(-1.5, 1.5)

# Single legend in the last panel
axes[-1].legend(fontsize=7.5, loc='lower right', title='Control shapes')
fig.suptitle('Four-control-shape SBS — effect of locality parameter α', fontsize=13)
fig.tight_layout()
plt.show()


The slider below lets you adjust **α** (locality) interactively.  
- Drag **α** towards 0 to see global averaging of all four shapes.  
- Increase **α** to strengthen each shape's local influence and watch individual features emerge.


In [ ]:
t_dense = np.linspace(0, 1, 700)

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    out_4cp = widgets.Output()
    alpha_4cp = widgets.FloatSlider(
        value=2.0, min=0.1, max=10.0, step=0.1,
        description='α (locality)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='50%'),
    )

    def redraw_4cp(change=None):
        with out_4cp:
            clear_output(wait=True)
            sbs = ShapeBlendSpline(ctrl_shapes, locality=alpha_4cp.value)
            pts = sbs.evaluate(t_dense)

            fig, ax = plt.subplots(figsize=(6, 5.5))
            for s, lbl, col in zip(ctrl_shapes, ctrl_labels, ctrl_colors):
                sp = s(t_dense)
                ax.plot(sp[:,0], sp[:,1], '--', color=col, lw=1.2, alpha=0.5, label=lbl)
            ax.plot(pts[:,0], pts[:,1], 'k-', lw=2.8, zorder=5, label='SBS blend')
            ax.set_aspect('equal')
            ax.set_title(f'Four-control-shape SBS   α = {alpha_4cp.value:.1f}', fontsize=12)
            ax.set_xlabel('x'); ax.set_ylabel('y')
            ax.set_xlim(-1.9, 1.9); ax.set_ylim(-1.5, 1.5)
            ax.legend(fontsize=8)
            fig.tight_layout()
            plt.show()

    alpha_4cp.observe(redraw_4cp, names='value')
    redraw_4cp()
    display(widgets.VBox([alpha_4cp, out_4cp]))

except ImportError:
    print('ipywidgets not available — showing static example instead.')
    sbs = ShapeBlendSpline(ctrl_shapes, locality=2.0)
    pts = sbs.evaluate(t_dense)
    fig, ax = plt.subplots(figsize=(6, 5.5))
    for s, lbl, col in zip(ctrl_shapes, ctrl_labels, ctrl_colors):
        sp = s(t_dense)
        ax.plot(sp[:,0], sp[:,1], '--', color=col, lw=1.2, alpha=0.5, label=lbl)
    ax.plot(pts[:,0], pts[:,1], 'k-', lw=2.8, label='SBS blend')
    ax.set_aspect('equal'); ax.set_title('Four-control-shape SBS (α=2.0)')
    ax.legend(); plt.show()


---
## 2  Multi-shape Shape Blend Spline

Blend **five shapes** — circle, ellipse, superellipse, rectangle, star — using the partition-of-unity framework.

The panels below compare **three blending strategies** side-by-side:

1. **Naïve global blend** (`ShapeBlender`, equal weights): every shape contributes at every parameter value → features are washed out into a featureless averaged curve.
2. **SBS at moderate α (= 2)**: each shape begins to dominate near its own centre parameter → feature transitions become visible.
3. **SBS at high α (= 6)**: strong locality → each zone of the curve closely tracks its constituent shape, preserving distinct geometric features.

The key advantage of SBS over naïve blending is **controllable feature preservation**: you decide, via α, how much of each shape's character is retained in the blend.


In [ ]:
# ── Define 5 shapes (all centred at origin, comparable scale) ───────────
shapes_5 = [
    partial(circle_arc,       center=(0,0), radius=1.0),
    partial(ellipse_arc,      center=(0,0), a=1.4, b=0.7),
    partial(superellipse_arc, center=(0,0), a=1.1, b=1.1, exponent=4.0),
    partial(rectangle_arc,    center=(0,0), width=1.6, height=1.2),
    partial(star_arc,         center=(0,0), outer_radius=1.2, inner_radius=0.45, n_points=5),
]
labels_5 = ['Circle', 'Ellipse', 'Superellipse (n=4)', 'Rectangle', 'Star']
colors_5 = list(plt.cm.tab10.colors)

t_plot = np.linspace(0, 1, 600)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Panel 1: naïve global blend (ShapeBlender, equal weights) ────────────
blender_eq = ShapeBlender(shapes_5)
pts_naive  = blender_eq.evaluate(t_plot)
for j, (s, lbl) in enumerate(zip(shapes_5, labels_5)):
    sp = s(t_plot)
    axes[0].plot(sp[:,0], sp[:,1], '--', color=colors_5[j], lw=1, alpha=0.4, label=lbl)
axes[0].plot(pts_naive[:,0], pts_naive[:,1], 'k-', lw=2.8, label='Naïve blend')
axes[0].set_aspect('equal')
axes[0].set_title('Naïve global blend\n(features averaged away)', fontsize=10)
axes[0].legend(fontsize=7, ncol=2); axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

# ── Panels 2 & 3: SBS at moderate and high locality ──────────────────────
panel_cfg = [
    (axes[1], 2.0, 'SBS  α = 2  (moderate locality)\n(feature transitions visible)'),
    (axes[2], 6.0, 'SBS  α = 6  (high locality)\n(each zone tracks its shape)'),
]
for ax, alpha, title in panel_cfg:
    sbs = blend_shape_series(shapes_5, locality=alpha)
    pts = sbs.evaluate(t_plot)
    for j, s in enumerate(shapes_5):
        sp = s(t_plot)
        ax.plot(sp[:,0], sp[:,1], '--', color=colors_5[j], lw=1, alpha=0.4)
    ax.plot(pts[:,0], pts[:,1], 'k-', lw=2.8, label=f'SBS  α={alpha}')
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8); ax.set_xlabel('x'); ax.set_ylabel('y')

fig.suptitle('Multi-shape Shape Blend Spline: naïve blend vs SBS', fontsize=13)
fig.tight_layout()
plt.show()


### 2a  Interactive multi-shape SBS

Use the **α** slider to tune locality (left panel: SBS).  
Use the **weight** sliders to promote or suppress individual shapes in the naïve weighted blend (right panel).  
Notice how the SBS (left) retains distinct shape features even at modest α, while the naïve blend (right) only shows a smooth intermediate.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    out_multi = widgets.Output()

    alpha_slider = widgets.FloatSlider(
        value=2.0, min=0.1, max=8.0, step=0.1,
        description='α (locality)', style={'description_width': 'initial'},
        layout=widgets.Layout(width='55%'),
    )
    w0 = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description='Circle')
    w1 = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description='Ellipse')
    w2 = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description='Superell.')
    w3 = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description='Rectangle')
    w4 = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description='Star')

    def redraw_multi(change=None):
        with out_multi:
            clear_output(wait=True)
            raw_w = np.array([w0.value, w1.value, w2.value, w3.value, w4.value])
            total = raw_w.sum()
            norm_w = raw_w / max(total, 1e-9)

            sbs     = blend_shape_series(shapes_5, locality=alpha_slider.value)
            blender = ShapeBlender(shapes_5, weights=raw_w)
            pts_sbs = sbs.evaluate(t_plot)
            pts_mix = blender.evaluate(t_plot)

            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            # Left: SBS with locality
            for j, (s, lbl) in enumerate(zip(shapes_5, labels_5)):
                sp = s(t_plot)
                axes[0].plot(sp[:,0], sp[:,1], '--', color=colors_5[j], lw=1,
                             alpha=0.30, label=lbl)
            axes[0].plot(pts_sbs[:,0], pts_sbs[:,1], 'k-', lw=2.5, label='SBS blend')
            axes[0].set_aspect('equal')
            axes[0].set_title(f'SBS  α = {alpha_slider.value:.1f}\n(locality-preserving blend)')
            axes[0].legend(fontsize=7, ncol=2); axes[0].set_xlabel('x'); axes[0].set_ylabel('y')

            # Right: global weighted blend
            for j, (s, lbl) in enumerate(zip(shapes_5, labels_5)):
                sp = s(t_plot)
                axes[1].plot(sp[:,0], sp[:,1], '--', color=colors_5[j], lw=1,
                             alpha=0.30, label=f'{lbl} (w={norm_w[j]:.2f})')
            axes[1].plot(pts_mix[:,0], pts_mix[:,1], 'k-', lw=2.5, label='Weighted blend')
            axes[1].set_aspect('equal')
            wstr = '[' + ', '.join(f'{x:.2f}' for x in norm_w) + ']'
            axes[1].set_title(f'Naïve global blend\nweights {wstr}')
            axes[1].legend(fontsize=7, ncol=2); axes[1].set_xlabel('x')

            fig.suptitle('SBS vs naïve global blend — use sliders to compare', fontsize=11)
            fig.tight_layout()
            plt.show()

    for slider in [alpha_slider, w0, w1, w2, w3, w4]:
        slider.observe(redraw_multi, names='value')

    redraw_multi()
    display(widgets.VBox([
        alpha_slider,
        widgets.HBox([w0, w1]),
        widgets.HBox([w2, w3, w4]),
        out_multi,
    ]))

except ImportError:
    print('ipywidgets not available — showing static plot instead.')
    blender = ShapeBlender(shapes_5)
    pts = blender.evaluate(t_plot)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(pts[:,0], pts[:,1], 'k-', lw=2.5)
    ax.set_aspect('equal'); ax.set_title('Equal-weight 5-shape blend'); plt.show()


---
## 3  Visualising the blend weights

The **partition-of-unity** property ensures the weights always sum to 1.  
Higher α concentrates each weight function near its centre parameter.


In [ ]:
t_w = np.linspace(0, 1, 400)
centers = np.linspace(0, 1, 5)
labels_w = ['Circle', 'Ellipse', 'Superell.', 'Rectangle', 'Star']

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, alpha in zip(axes, [0.5, 1.5, 5.0]):
    W = blend_weights(t_w, centers, locality=alpha)
    for j in range(5):
        ax.plot(t_w, W[j], color=colors_5[j], lw=2, label=labels_w[j])
    ax.fill_between(t_w, 0, 1, alpha=0.04)
    ax.set_ylim(-0.02, 1.05); ax.set_xlabel('t'); ax.set_ylabel('W_j(t)')
    ax.set_title(f'Blend weights  α = {alpha}')
    if alpha == 0.5:
        ax.legend(fontsize=8, loc='upper right')
fig.suptitle('Partition-of-unity blend weights at different locality α', fontsize=12)
fig.tight_layout()
plt.show()
print("Note: columns always sum to 1.0 (partition of unity)")
print("Max deviation from 1:", np.abs(W.sum(axis=0) - 1).max())

---
## 4  Shape morphing sequence

Smoothly morph from one shape to another by varying the blend parameter β ∈ [0, 1].


In [ ]:
n_frames = 7
betas = np.linspace(0, 1, n_frames)
frames = shape_morph(circle_arc, star_arc, n_frames=n_frames, locality=2.5, n_points=400)

fig, axes = plt.subplots(1, n_frames, figsize=(3.2*n_frames, 3.5))
for ax, pts, beta in zip(axes, frames, betas):
    ax.plot(pts[:,0], pts[:,1], color='steelblue', lw=2.0)
    ax.set_aspect('equal'); ax.set_title(f'β={beta:.2f}', fontsize=10)
    ax.axis('off')
fig.suptitle('Shape morphing: circle → star (β = blend parameter)', fontsize=12)
fig.tight_layout()
plt.show()

---
## 5  Free-form curve from control points

Specify your own (x, y) **control points** and get a smooth Shape Blend Spline automatically.  
Try editing the `control_pts` array in the next cell!


In [ ]:
from shape_blend_splines.curve import ControlPointSpline

# ── Edit these control points ─────────────────────────────────────────────
control_pts = np.array([
    [0.0,  0.0],
    [1.0,  1.5],
    [2.5,  0.4],
    [3.5,  2.0],
    [5.0,  0.0],
    [6.0,  1.2],
])
locality_cp = 2.0   # try 0.5, 1.0, 3.0, …
# ─────────────────────────────────────────────────────────────────────────

sbs_cp = ControlPointSpline(control_pts, locality=locality_cp)
t_cp   = np.linspace(0, 1, 600)
pts_cp = sbs_cp.evaluate(t_cp)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pts_cp[:,0], pts_cp[:,1], color='steelblue', lw=2.5, label='Shape Blend Spline')
ax.plot(control_pts[:,0], control_pts[:,1], 'o--', color='tomato',
        lw=1.5, markersize=9, label='Control points')
for i, (x, y) in enumerate(control_pts):
    ax.annotate(f'P{i}', (x, y), textcoords='offset points', xytext=(4, 6), fontsize=9)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'Free-form control-point spline  (α = {locality_cp})')
ax.legend(); fig.tight_layout(); plt.show()

### 5a  Interactive control-point editor

Drag the **α (locality)** slider and the per-point **y-offset** sliders to reshape the curve interactively.

**How α works here:** the control-point editor represents the curve as a `ShapeBlendSpline` with one *line-segment shape* per pair of consecutive control points (N − 1 shapes total for N points).  
Each segment is a separate SBS shape centred at its own parameter position:  
- **Low α** → smooth global interpolation, curve flows naturally through all points.  
- **High α** → strong locality, curve follows each linear segment closely with tight transitions at the control points.

(For full click-to-drag interaction, run the notebook locally with `%matplotlib widget`.)


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    from shape_blend_splines.shapes import line_segment as _line_seg

    out_cp = widgets.Output()

    # Base control points (x fixed, y adjustable via sliders)
    base_pts = np.array([
        [0.0, 0.0], [1.0, 1.5], [2.5, 0.4],
        [3.5, 2.0], [5.0, 0.0], [6.0, 1.2],
    ])

    alpha_cp_s = widgets.FloatSlider(
        value=2.0, min=0.1, max=8.0, step=0.1,
        description='α (locality)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='55%'),
    )
    dy_sliders = [
        widgets.FloatSlider(value=base_pts[i, 1], min=-3.0, max=4.0, step=0.05,
                            description=f'P{i}.y')
        for i in range(len(base_pts))
    ]

    def _make_seg_sbs(pts, locality):
        """Per-segment ShapeBlendSpline: one line-shape per consecutive pair.

        With a single shape (as ControlPointSpline does by default), the
        locality parameter α has no effect because the blend weight is always
        1.0. Using N-1 segment shapes restores α's visual effect.
        """
        n = len(pts)
        seg_shapes = [
            partial(_line_seg, p0=pts[i].copy(), p1=pts[i + 1].copy())
            for i in range(n - 1)
        ]
        t_centers = np.linspace(0.0, 1.0, n - 1)
        return ShapeBlendSpline(seg_shapes, t_centers=t_centers, locality=locality)

    def redraw_cp(change=None):
        with out_cp:
            clear_output(wait=True)
            pts_edit = base_pts.copy()
            for i, sl in enumerate(dy_sliders):
                pts_edit[i, 1] = sl.value

            sbs_e = _make_seg_sbs(pts_edit, locality=alpha_cp_s.value)
            t_e   = np.linspace(0, 1, 600)
            crv   = sbs_e.evaluate(t_e)

            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(crv[:,0], crv[:,1], color='steelblue', lw=2.5, label='SBS curve')
            ax.plot(pts_edit[:,0], pts_edit[:,1], 'o--', color='tomato',
                    lw=1.5, markersize=9, label='Control pts')
            for i, (x, y) in enumerate(pts_edit):
                ax.annotate(f'P{i}', (x, y),
                            textcoords='offset points', xytext=(4, 6), fontsize=8)
            ax.set_xlabel('x'); ax.set_ylabel('y')
            ax.set_title(f'Interactive control-point SBS  α = {alpha_cp_s.value:.1f}')
            ax.legend(); ax.set_ylim(-3.5, 5); fig.tight_layout(); plt.show()

    alpha_cp_s.observe(redraw_cp, names='value')
    for sl in dy_sliders:
        sl.observe(redraw_cp, names='value')

    redraw_cp()
    slider_rows = [widgets.HBox(dy_sliders[:3]), widgets.HBox(dy_sliders[3:])]
    display(widgets.VBox([alpha_cp_s] + slider_rows + [out_cp]))

except ImportError:
    print('ipywidgets not available. Edit `control_pts` manually in the cell above.')


---
## 6  Defining your own shape

Any callable `shape(t) -> np.ndarray of shape (m, 2)` can be used.  
Here we blend a **Lissajous figure** with a **circle**.


In [ ]:
def lissajous(t, a=1.0, b=2.0, delta=np.pi/4):
    """Lissajous figure: x = a*cos(t*2π), y = b*sin(2*t*2π + δ)."""
    theta = t * 2 * np.pi
    x = a * np.cos(theta)
    y = b * np.sin(2 * theta + delta)
    return np.column_stack([x, y])

t_adv = np.linspace(0, 1, 600)
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, beta in zip(axes, [0.0, 0.33, 0.67, 1.0]):
    blender = ShapeBlender([circle_arc, lissajous], weights=[1-beta, beta])
    pts_adv = blender.evaluate(t_adv)
    ax.plot(circle_arc(t_adv)[:,0],  circle_arc(t_adv)[:,1],
            '--', color='gray', lw=1, alpha=0.3, label='Circle')
    ax.plot(lissajous(t_adv)[:,0],   lissajous(t_adv)[:,1],
            ':', color='gray', lw=1, alpha=0.3, label='Lissajous')
    ax.plot(pts_adv[:,0], pts_adv[:,1], color='darkorchid', lw=2.5, label=f'β={beta:.2f}')
    ax.set_aspect('equal'); ax.set_title(f'β = {beta:.2f}'); ax.legend(fontsize=7)
    ax.set_xlabel('x'); ax.set_ylabel('y')

fig.suptitle('Custom shape blend: circle → Lissajous figure', fontsize=12)
fig.tight_layout()
plt.show()

---
## Summary

| Feature | API |
|---------|-----|
| Four-point shape family | `ControlPointSpline(pts, locality=α)` |
| Multi-shape SBS | `ShapeBlendSpline([S1, S2, …], locality=α)` |
| Weight-only blend | `ShapeBlender([S1, S2, …], weights=[w1, w2, …])` |
| Control-point curve | `ControlPointSpline(pts, locality=α)` |
| Shape morphing | `shape_morph(S_a, S_b, n_frames=N)` |

**Key parameter — locality α (for SBS / control-point sections):**
- Low α (≈ 0.5) → diffuse global blending  
- α = 1 → smooth raised-cosine blending (default)  
- High α (≥ 3) → strong local shape preservation near each centre

**Cite:**  
Q. Li, "Shape Blend Splines", *Computer-Aided Design*, vol. 43, no. 8, 2011.  
[DOI 10.1016/j.cad.2011.01.006](https://doi.org/10.1016/j.cad.2011.01.006)
